[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hentablihamza/practical-molecular-dynamics-course/blob/main/notebooks/06_preparing_a_protein.ipynb)

# 06 · Preparing a Protein: From PDB File to Simulation-Ready Structure

*Practical Molecular Dynamics Course · Hamza Hentabli, Ph.D. · ~50 min*

A PDB file is an experimental *model*, not a simulation input. X-ray structures usually lack hydrogens, may
miss side-chain atoms or whole loops, contain crystallization additives, and say nothing about protonation
states. In this notebook we prepare two real structures with **PDBFixer**: the tiny β-hairpin **chignolin
(CLN025, PDB 5AWL)** that we will simulate in Notebooks 07–09, and the EGFR kinase–erlotinib complex
(**PDB 1M17**) that shows the problems you meet with real drug targets.

**Learning objectives.** By the end of this notebook you will be able to:

- read a PDB entry critically: method, resolution, missing residues and atoms, heterogens
- use PDBFixer to add missing atoms, remove unwanted heterogens, and add hydrogens at a chosen pH
- explain protonation states and see how pH changes the net charge of a protein
- decide what to do with missing loops and termini, and save a clean structure

In [1]:
# Setup: installs dependencies on Google Colab and locates the course data.
# Running locally? Create the environment from environment.yml once and this cell does nothing extra.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'openmm', 'pdbfixer', 'py3Dmol'], check=True)
    if not os.path.isdir("practical-molecular-dynamics-course"):
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/hentablihamza/practical-molecular-dynamics-course.git"], check=True)
    DATA_DIR = "practical-molecular-dynamics-course/data"
else:
    DATA_DIR = "../data" if os.path.isdir("../data") else "data"

print("Running on Colab:", IN_COLAB, "| data folder:", DATA_DIR)

Running on Colab: False | data folder: ../data


## 1. Get the structure (online, with offline fallback)

**Chignolin CLN025** is a 10-residue designed peptide (YYDPETGTWY) that folds into a stable β-hairpin in water
in microseconds, which makes it a favourite test system for MD (Honda *et al.*, *J. Am. Chem. Soc.* 2008).
PDB 5AWL is its 1.11 Å crystal structure.

In [2]:
import os, urllib.request
import numpy as np
import pandas as pd
import openmm
from openmm import app, unit
from pdbfixer import PDBFixer

OFFLINE = os.environ.get("COURSE_OFFLINE") == "1"
os.makedirs("output", exist_ok=True)

def get_pdb(pdb_id):
    path = f"output/{pdb_id}.pdb"
    if not OFFLINE:
        try:
            urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id}.pdb", path)
            print("downloaded from RCSB:", path)
            return path
        except Exception as err:
            print("download failed:", err)
    path = f"{DATA_DIR}/structures/{pdb_id}.pdb"
    print("using cached copy:", path)
    return path

path = get_pdb("5AWL")
header = [l.rstrip() for l in open(path) if l.startswith(("TITLE", "EXPDTA")) or "RESOLUTION." in l]
print("\n".join(header))

using cached copy: ../data/structures/5AWL.pdb
TITLE     CRYSTAL STRUCTURE OF A MUTANT OF CHIGNOLIN, CLN025
EXPDTA    X-RAY DIFFRACTION
REMARK   2 RESOLUTION.    1.11 ANGSTROMS.


In [3]:
def describe(topology):
    elements = pd.Series([a.element.symbol for a in topology.atoms()]).value_counts().to_dict()
    residues = pd.Series([r.name for r in topology.residues()]).value_counts().to_dict()
    return {"atoms": topology.getNumAtoms(), "residues": topology.getNumResidues(), "elements": elements, "residue names": residues}

raw = app.PDBFile(path)
describe(raw.topology)

{'atoms': 105,
 'residues': 22,
 'elements': {'C': 62, 'O': 32, 'N': 11},
 'residue names': {'HOH': 12,
  'TYR': 3,
  'THR': 2,
  'ASP': 1,
  'GLU': 1,
  'PRO': 1,
  'GLY': 1,
  'TRP': 1}}

The file has heavy atoms only (X-ray rarely sees hydrogens), plus crystal waters. A force field needs every atom.

## 2. PDBFixer, step by step

PDBFixer reads the file and the `SEQRES` records, compares them, and lets us decide what to fix. Each `find…`
step only *reports*; the `add…`/`remove…` steps change the structure.

In [4]:
fixer = PDBFixer(filename=path)
fixer.findMissingResidues()
fixer.findNonstandardResidues()
fixer.findMissingAtoms()
print("missing residues      :", fixer.missingResidues)
print("non-standard residues :", fixer.nonstandardResidues)
print("missing heavy atoms   :", {f"{r.name}{r.id}": [a.name for a in atoms] for r, atoms in fixer.missingAtoms.items()})
print("missing terminals     :", {f"{r.name}{r.id}": t for r, t in fixer.missingTerminals.items()})

fixer.removeHeterogens(keepWater=False)     # drop crystal waters (we will add a full water box later)
fixer.addMissingAtoms()
n_before = fixer.topology.getNumAtoms()
fixer.addMissingHydrogens(pH=7.0)
print(f"\natoms: {n_before} heavy atoms -> {fixer.topology.getNumAtoms()} with hydrogens at pH 7.0")

missing residues      : {}
non-standard residues : []
missing heavy atoms   : {}
missing terminals     : {}

atoms: 93 heavy atoms -> 166 with hydrogens at pH 7.0


Chignolin is a clean structure: every residue and heavy atom is present. The only real work was hydrogens.

## 3. Protonation states depend on pH

Which hydrogens are added is a chemical decision. At pH 7, Asp and Glu carboxylates are deprotonated (−1), Lys
and Arg are protonated (+1), and the N- and C-termini are charged. His (pKa ≈ 6) is the tricky one and can be
neutral (HID/HIE) or charged (HIP). OpenMM's `Modeller.addHydrogens` uses standard pKa values and reports the
variant it chose for each residue. Let us compare the net charge of chignolin at several pH values.

In [5]:
ff = app.ForceField("amber14-all.xml", "amber14/tip3pfb.xml")

def protonation(residue):
    # Name the protonation state from the hydrogens that are present (AMBER names).
    names = {a.name for a in residue.atoms()}
    if residue.name == "ASP": return "ASH (neutral)" if "HD2" in names else "ASP (-1)"
    if residue.name == "GLU": return "GLH (neutral)" if "HE2" in names else "GLU (-1)"
    if residue.name == "LYS": return "LYS (+1)" if "HZ3" in names else "LYN (neutral)"
    if residue.name == "HIS":
        return {(True, True): "HIP (+1)", (True, False): "HID", (False, True): "HIE"}[("HD1" in names, "HE2" in names)]
    if residue.name == "TYR": return "TYR" if "HH" in names else "TYM (-1)"
    return None

def net_charge(topology, positions, pH):
    m = app.Modeller(topology, positions)
    m.addHydrogens(ff, pH=pH)
    system = ff.createSystem(m.topology)
    nbf = [f for f in system.getForces() if isinstance(f, openmm.NonbondedForce)][0]
    q = sum(nbf.getParticleParameters(i)[0].value_in_unit(unit.elementary_charge) for i in range(system.getNumParticles()))
    chosen = {f"{r.name}{r.id}": protonation(r) for r in m.topology.residues() if protonation(r)}
    return round(q, 3), m.topology.getNumAtoms(), chosen

heavy = PDBFixer(filename=path); heavy.removeHeterogens(False); heavy.findMissingResidues()
heavy.findMissingAtoms(); heavy.addMissingAtoms()
rows = []
for pH in [2.0, 4.0, 7.0, 11.0]:
    q, n, chosen = net_charge(heavy.topology, heavy.positions, pH)
    rows.append({"pH": pH, "net charge": q, "atoms": n, "variants": chosen})
pd.set_option("display.max_colwidth", 200)
pd.DataFrame(rows)

,pH,net charge,atoms,variants
0,2.0,0.0,168,"{'TYR1': 'TYR', 'TYR2': 'TYR', 'ASP3': 'ASH (neutral)', 'GLU5': 'GLH (neutral)', 'TYR10': 'TYR'}"
1,4.0,0.0,168,"{'TYR1': 'TYR', 'TYR2': 'TYR', 'ASP3': 'ASH (neutral)', 'GLU5': 'GLH (neutral)', 'TYR10': 'TYR'}"
2,7.0,-2.0,166,"{'TYR1': 'TYR', 'TYR2': 'TYR', 'ASP3': 'ASP (-1)', 'GLU5': 'GLU (-1)', 'TYR10': 'TYR'}"
3,11.0,-2.0,166,"{'TYR1': 'TYR', 'TYR2': 'TYR', 'ASP3': 'ASP (-1)', 'GLU5': 'GLU (-1)', 'TYR10': 'TYR'}"


CLN025 has two acidic side chains (Asp3, Glu5) and charged termini (+1 and −1). At pH 7 both carboxylates are
deprotonated and the net charge is −2. At low pH they are protonated (ASH, GLH) and the peptide becomes
neutral overall (the standard C-terminal template stays charged, balancing the N-terminus). Note that
OpenMM's simple rule only compares the pH with a standard pKa for each residue type; it knows nothing about
the local environment. For real proteins, especially in **binding sites**, use a pKa predictor (PROPKA, H++, PDB2PQR)
and your chemical judgement: a wrong histidine or aspartate state in a pocket can ruin a docking or
binding-free-energy study.

## 4. A messier, drug-relevant case: EGFR kinase with erlotinib (PDB 1M17)

In [6]:
path_1m17 = get_pdb("1M17")
fx = PDBFixer(filename=path_1m17)
fx.findMissingResidues()
fx.findNonstandardResidues()
fx.findMissingAtoms()
chain_res = {c.index: [r for r in c.residues()] for c in fx.topology.chains()}
print("chains (id, residues):", [(c.id, len(list(c.residues()))) for c in fx.topology.chains()], "<- protein, then ligand + waters")
print("first/last residue in the file:", chain_res[0][0].name + chain_res[0][0].id, "/", chain_res[0][-1].name + chain_res[0][-1].id)
for (chain_i, pos), names in fx.missingResidues.items():
    after = chain_res[chain_i][pos - 1].id if pos > 0 else "N-terminus"
    print(f"missing {len(names):2d} residues after {after}: {'-'.join(names)}")
het = pd.Series([r.name for r in fx.topology.residues() if r.name not in app.Modeller._residueHydrogens
                 and r.name not in ("HOH",)]).value_counts()
print("heterogens (non-standard, non-water):", het.to_dict())
print("missing heavy atoms:", {f"{r.name}{r.id}": [a.name for a in atoms] for r, atoms in fx.missingAtoms.items()} or "none")

using cached copy: ../data/structures/1M17.pdb
chains (id, residues): [('A', 312), ('A', 21)] <- protein, then ligand + waters
first/last residue in the file: GLY672 / PRO995
missing  6 residues after N-terminus: GLY-SER-HIS-MET-ALA-SER
missing 12 residues after 964: LEU-PRO-SER-PRO-THR-ASP-SER-ASN-PHE-TYR-ARG-ALA
missing  3 residues after 995: GLN-GLN-GLY
heterogens (non-standard, non-water): {'AQ4': 1}
missing heavy atoms: none


Typical decisions for this structure:

- **Erlotinib (AQ4)** is the drug: keep it, but it needs its own force-field parameters (Notebook 11).
  PDBFixer's `removeHeterogens` would delete it, so ligands are handled separately.
- **Missing N-terminal tag (GSHMAS)** comes from the expression construct: do not build it.
- **Missing internal segment (residues 965–976)** in the C-terminal tail: either model it (PDBFixer can build
  a crude loop, which then needs careful equilibration) or trim the chain before the gap. For a short
  simulation focused on the binding site, trimming the tail is the cleaner choice, and it is what Notebook 11 does.
- **Crystal waters**: often removed and replaced by a full water box; key waters in a binding site are sometimes kept.

Here is how building a loop would look, so you can see what PDBFixer does (we will not use this model):

In [7]:
fx_loop = PDBFixer(filename=path_1m17)
fx_loop.removeHeterogens(keepWater=False)
fx_loop.findMissingResidues()
chains = list(fx_loop.topology.chains())
# keep only internal gaps: drop entries at the very start or end of a chain
fx_loop.missingResidues = {k: v for k, v in fx_loop.missingResidues.items()
                           if 0 < k[1] < len(list(chains[k[0]].residues()))}
fx_loop.findMissingAtoms()
fx_loop.addMissingAtoms()
print("residues to build:", {k: len(v) for k, v in fx_loop.missingResidues.items()})
print("protein residues after loop building:", fx_loop.topology.getNumResidues())

residues to build: {(0, 293): 12}
protein residues after loop building: 324


## 5. Look at the prepared chignolin and save it

In [8]:
app.PDBFile.writeFile(fixer.topology, fixer.positions, open("output/chignolin_prepared.pdb", "w"))
print(describe(fixer.topology))

try:
    import py3Dmol
    view = py3Dmol.view(width=500, height=350)
    view.addModel(open("output/chignolin_prepared.pdb").read(), "pdb")
    view.setStyle({"stick": {"colorscheme": "greenCarbon", "radius": 0.15}})
    view.addStyle({"elem": "H"}, {"stick": {"color": "white", "radius": 0.08}})
    view.zoomTo(); view.show()
except ImportError:
    print("py3Dmol is not installed; open output/chignolin_prepared.pdb in PyMOL, ChimeraX, or VMD.")

{'atoms': 166, 'residues': 10, 'elements': {'H': 73, 'C': 62, 'O': 20, 'N': 11}, 'residue names': {'TYR': 3, 'THR': 2, 'ASP': 1, 'PRO': 1, 'GLU': 1, 'GLY': 1, 'TRP': 1}}


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

**Preparation checklist** (use it for every new system):

1. Method and resolution? NMR ensembles: choose a model. Alternate locations: choose a conformer.
2. Missing residues and atoms? Build short internal gaps, trim or cap long ones; do not build tags.
3. Non-standard residues (MSE → MET, modified residues)? `replaceNonstandardResidues()`.
4. Ligands, ions, cofactors, waters: keep, parameterize, or remove, deliberately.
5. Protonation at the experimental pH, with extra care for His, Asp, Glu, Cys, and binding-site residues.
6. Disulfide bonds (Cys–Cys within ~0.2 nm) are detected by OpenMM from the geometry; check them.
7. Save the prepared structure and **write down every decision** (it belongs in your methods section).

### ✏️ Exercise 1

NMR structures contain many models. Load the original chignolin NMR structure **1UAO** (cached in
`data/structures/`) with `app.PDBFile` and print how many models (`getNumFrames()`) and atoms it has. Does it
already contain hydrogens?

In [9]:
# Your code here

<details>
<summary><b>Show solution</b></summary>

```python
nmr = app.PDBFile(f"{DATA_DIR}/structures/1UAO.pdb")
print("models:", nmr.getNumFrames(), "| atoms:", nmr.topology.getNumAtoms())
print("hydrogens:", sum(a.element.symbol == "H" for a in nmr.topology.atoms()))
```

</details>

### ✏️ Exercise 2

Prepare the 1M17 protein at **pH 5.0** and at **pH 7.0** (remove heterogens and do not build missing residues)
and print the histidine variants that OpenMM chooses at each pH. (Hint: reuse `net_charge` on a fixer with
`missingResidues = {}`.)

In [10]:
# Your code here

<details>
<summary><b>Show solution</b></summary>

```python
fx2 = PDBFixer(filename=path_1m17); fx2.removeHeterogens(False)
fx2.findMissingResidues(); fx2.missingResidues = {}
fx2.findMissingAtoms(); fx2.addMissingAtoms()
for pH in (5.0, 7.0):
    q, n, chosen = net_charge(fx2.topology, fx2.positions, pH)
    print("pH", pH, "| charge", q, "|", {k: v for k, v in chosen.items() if k.startswith("HIS")})
```

</details>

### ✏️ Exercise 3

How many crystal waters does 1M17 contain, and how many of them are within 0.35 nm of any erlotinib (AQ4)
atom? Use `raw = app.PDBFile(path_1m17)` and NumPy distances.

In [11]:
# Your code here

<details>
<summary><b>Show solution</b></summary>

```python
raw2 = app.PDBFile(path_1m17)
X = np.array(raw2.positions.value_in_unit(unit.nanometer))
lig = [a.index for a in raw2.topology.atoms() if a.residue.name == "AQ4"]
wat = [a.index for a in raw2.topology.atoms() if a.residue.name == "HOH"]
d = np.linalg.norm(X[wat][:, None] - X[lig][None], axis=-1).min(1)
print(len(wat), "waters;", int((d < 0.35).sum()), "within 0.35 nm of erlotinib")
```

</details>

## Key takeaways

- A PDB entry needs preparation: hydrogens, missing atoms, heterogens, protonation states.
- PDBFixer finds problems first, then fixes only what you ask for.
- pH sets protonation states and the net charge; binding-site residues deserve a pKa check.
- Do not build expression tags; build short internal loops or trim; handle ligands separately.

**Next:** [07 · Solvation, ions, and energy minimization](07_solvation_minimization.ipynb).